In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle as pkl
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

In [ ]:
NUM_COL = 'true_prc'
DATE_COL = 'dlycaldt'
TICKER_COL1 = 'permno'
TICKER_COL2 = 'ticker'
SPLIT_DATE = "2021-01-01"
START_BALANCE = 1000
EXIT_PROP = 1.0
LOOKBACK = 3
n_train_tickers= 1000

In [ ]:
data = pd.read_csv('./data/crsp_dsf.csv')
data[DATE_COL] = data[DATE_COL].astype('datetime64[s]')
data = data.dropna(subset='dlyclose').reset_index(drop=True)
data['dlyvol'] = data['dlyvol'].fillna(0)
data['tot_shares'] = (data['dlycap'] / data['dlyprc']).round(0)
data[NUM_COL] = data['dlyclose'] * data['tot_shares'] / 1000000
data.head()

In [ ]:
avsp = pd.read_csv(f'./data/all_avsp_crsp_window_{LOOKBACK}.csv')
avsp[DATE_COL] = pd.to_datetime(avsp[DATE_COL])

In [ ]:
added_cols = ['pos', 'neg']
avsp_cols = [TICKER_COL1, TICKER_COL2, DATE_COL]
avsp_cols.extend(added_cols)

if 'pos' in data.columns:
    data.drop(columns=added_cols, inplace=True)

data = pd.merge(left=data, right=avsp[avsp_cols],  on=[TICKER_COL1, TICKER_COL2, DATE_COL])

# Delete avsp after merge to save memory (we don't need it after this)
del avsp

In [ ]:
# tickers = data[[TICKER_COL1, TICKER_COL2]].drop_duplicates().reset_index(drop=True)

# training_tickers = tickers.sample(n=n_train_tickers,random_state=29)

# training_tickers.to_csv("./data/training_tickers.csv", index=False)

# training_tickers = pd.read_csv("./data/training_tickers.csv")

In [ ]:
reg_df = data.sort_values([TICKER_COL1, TICKER_COL2, DATE_COL]).copy()

reg_df["target"] = (reg_df.groupby([TICKER_COL1, TICKER_COL2])[NUM_COL].shift(-1))

for lag in [1, 2, 3, 5, 10, 20]:
    reg_df[f"lag_prc_{lag}"] = (reg_df.groupby([TICKER_COL1, TICKER_COL2])[NUM_COL].shift(lag))

for lag in [1, 2, 3, 5, 10]:
    reg_df[f"lag_ret_{lag}"] = (reg_df.groupby([TICKER_COL1, TICKER_COL2])["dlyret"].shift(lag))

for win in [5, 10, 20, 50]:
    reg_df[f"ma_{win}"] = (reg_df.groupby([TICKER_COL1, TICKER_COL2])[NUM_COL].transform(lambda x: x.rolling(win).mean()))

reg_df["vol_20"] = (reg_df.groupby([TICKER_COL1, TICKER_COL2])["dlyret"].transform(lambda x: x.rolling(20).std()))

reg_df["vol_ma_20"] = (reg_df.groupby([TICKER_COL1, TICKER_COL2])["dlyprcvol"].transform(lambda x: x.rolling(20).mean()))

reg_df["vol_ratio"] = reg_df["dlyprcvol"] / reg_df["vol_ma_20"]

reg_df["spread"] = reg_df["dlyask"] - reg_df["dlybid"]

FEATURES = ["lag_prc_1", "lag_prc_2", "lag_prc_3", "lag_prc_5", "lag_prc_10", "lag_prc_20",
            "lag_ret_1", "lag_ret_2", "lag_ret_3", "lag_ret_5", "lag_ret_10",
            "ma_5", "ma_10", "ma_20", "ma_50", "vol_20", "vol_ratio", "spread", "pos", "neg", "dlyvol"]

reg_df = reg_df[[TICKER_COL1, TICKER_COL2, DATE_COL, NUM_COL, "target"] + FEATURES].dropna().reset_index(drop=True)

# ticker_set = set(map(tuple, training_tickers[[TICKER_COL1, TICKER_COL2]].to_numpy()))

# mask = reg_df[[TICKER_COL1, TICKER_COL2]]

train_df = reg_df[reg_df[DATE_COL] < SPLIT_DATE]

test_df = reg_df[reg_df[DATE_COL] >= SPLIT_DATE]

X_train = train_df[FEATURES]
y_train = train_df["target"]

X_test = test_df[FEATURES]
y_test = test_df["target"]

reg_df.head()

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

tf.keras.backend.clear_session()
tf.random.set_seed(29)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(32, activation="relu"),

    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae")
    ]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=200,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

preds = model.predict(X_test_scaled).flatten()

print("Test MSE:", mean_squared_error(y_test, preds))

model.save("./models/tf_baseline_avsp_reg_model_072826_01.keras")

# Save scaler
with open("./models/tf_baseline_avsp_reg_scaler_072826_01.pkl", "wb") as f:
    pkl.dump(scaler, f)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("Training and Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
preds_df = test_df[[DATE_COL, TICKER_COL1, TICKER_COL2, NUM_COL, "target"]].copy()
preds_df['pred'] = preds

In [ ]:
def trade_strat_single(df, curr_b=1000, exit_prop=1.0, pred_col='pred'):
    ### Defaults to starting balance of 1000 and full exits
    curr_vol = 0

    returns = []

    for _, row in df.iterrows():

        pred = row[pred_col]
        num = row[NUM_COL]

        ### Buy signal
        if pred >= num:
            ### Only buy if we have balance
            ### Otherwise we will just be holding the stock
            if curr_b > 0:
                curr_vol += curr_b / num
                curr_b = 0

        ### Sell signal
        else:
            ### Only sell if we have volume to sell
            if curr_vol > 0:
                shares_sold = exit_prop * curr_vol
                curr_b += shares_sold * num
                curr_vol -= shares_sold
                
        ### Current Return of the Strategy
        ### Cash Balance + Value of Held Stock
        returns.append(curr_b + curr_vol * num)
    
    return returns

In [ ]:
def trade_strat(data, tickers, curr_b=START_BALANCE, exit_prop=EXIT_PROP, pred_col='pred'):
    ret_df = pd.DataFrame()
    for t in tickers:
        df = data[(data[TICKER_COL1] == t[0]) & (data[TICKER_COL2] == t[1])].reset_index(drop=True)
        returns = trade_strat_single(df, curr_b=curr_b, exit_prop=exit_prop, pred_col=pred_col)
        start_vol = START_BALANCE / df[NUM_COL].iloc[0]
        hold_ret = start_vol * df[NUM_COL].values
        stg_df = (
            df[[TICKER_COL1, TICKER_COL2, DATE_COL]]
            .copy()
            .assign(
                strat_ret=returns,
                hold_ret=hold_ret
            )
        )
        if len(ret_df) == 0:
            ret_df = stg_df.copy()
        else:
            ret_df = pd.concat([ret_df, stg_df])
    return ret_df

In [ ]:
# filename = f"./data/tf_baseline_avsp_reg_model_returns_072826_01.csv"
# pred_tickers = preds_df[[TICKER_COL1, TICKER_COL2]].drop_duplicates().reset_index(drop=True)
# ret_df = trade_strat(preds_df, pred_tickers.values)
# ret_df.to_csv(filename, index=False)

# ### For training data
# filename = f"./data/tf_baseline_avsp_reg_model_train_returns_072826_01.csv"
# preds = model.predict(X_train_scaled).flatten()
# preds_df = train_df[[DATE_COL, TICKER_COL1, TICKER_COL2, NUM_COL, "target"]].copy()
# preds_df['pred'] = preds
# pred_tickers = preds_df[[TICKER_COL1, TICKER_COL2]].drop_duplicates().reset_index(drop=True)
# ret_df = trade_strat(preds_df, pred_tickers.values)
# ret_df.to_csv(filename, index=False)

filename = f"./data/tf_baseline_avsp_reg_model_returns_072826_01.csv"
ret_df = pd.read_csv(filename)
ret_df[DATE_COL] = pd.to_datetime(ret_df[DATE_COL])

In [ ]:
### There was some weird data corruption on the last day
plot_df = ret_df.groupby(by=DATE_COL).agg({"strat_ret" : "sum", "hold_ret" : "sum"}).iloc[:-1,:]

sns.lineplot(x=plot_df.index, y=np.log(plot_df['hold_ret']), label="Long Hold")
sns.lineplot(x=plot_df.index, y=np.log(plot_df['strat_ret']), label="Baseline Model Strategy")

plt.title("Portfolio Returns Comparison - Baseline Model Strategy v. Long Hold")
plt.ylabel("Log($ Return)")
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
### There was some weird data corruption on the last day
plot_df = ret_df.groupby(by=DATE_COL).agg({"strat_ret" : "sum", "hold_ret" : "sum", TICKER_COL1 : "count"}).rename(columns={TICKER_COL1:"num_stocks"}).iloc[:-1,:]
plot_df['base_bal'] = plot_df["num_stocks"] * START_BALANCE
plot_df["strat_pct_grwth"] = (plot_df["strat_ret"] / plot_df['base_bal'] - 1) * 100
plot_df["hold_pct_grwth"] = (plot_df["hold_ret"] / plot_df['base_bal'] - 1) * 100

sns.lineplot(x=plot_df.index, y=plot_df['hold_pct_grwth'], label="Long Hold")
sns.lineplot(x=plot_df.index, y=plot_df['strat_pct_grwth'], label="Baseline Model Strategy")

plt.title("Portfolio Returns Comparison - Baseline Model Strategy v. Long Hold")
plt.ylabel("% Return")
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
def objective(trial):

    tf.keras.backend.clear_session()

    model = tf.keras.Sequential()

    model.add(tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)))

    # Hidden layers
    n_layers = trial.suggest_int("n_layers", 1, 3)

    for i in range(n_layers):
        model.add(
            tf.keras.layers.Dense(
                units=trial.suggest_int(f"units_{i}", 32, 256, step=32),
                activation="relu"
            )
        )

        model.add(tf.keras.layers.BatchNormalization())

        model.add(
            tf.keras.layers.Dropout(
                trial.suggest_float(f"dropout_{i}", 0.0, 0.5)
            )
        )

    model.add(tf.keras.layers.Dense(1))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=trial.suggest_float(
                "learning_rate",
                1e-4,
                1e-2,
                log=True
            )
        ),
        loss="mse",
        metrics=["mae"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_test_scaled, y_test),
        epochs=200,
        batch_size=trial.suggest_categorical(
            "batch_size",
            [64, 128, 256, 512]
        ),
        callbacks=[early_stop],
        verbose=0
    )

    preds = model.predict(X_test_scaled, verbose=0).flatten()

    return mean_absolute_error(y_test, preds)

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

best_params = study.best_params

In [ ]:
best_params = study.best_params
best_params

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(29)

final_model = tf.keras.Sequential()

final_model.add(
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],))
)

for i in range(best_params["n_layers"]):
    final_model.add(
        tf.keras.layers.Dense(
            units=best_params[f"units_{i}"],
            activation="relu"
        )
    )

    final_model.add(tf.keras.layers.BatchNormalization())

    final_model.add(
        tf.keras.layers.Dropout(
            best_params[f"dropout_{i}"]
        )
    )

final_model.add(tf.keras.layers.Dense(1))

final_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=best_params["learning_rate"]
    ),
    loss="mse",
    metrics=["mae"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = final_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=20,
    batch_size=best_params["batch_size"],
    callbacks=[early_stop],
    verbose=1
)

final_preds = final_model.predict(X_test_scaled).flatten()

final_model.save("./models/tf_tuned_avsp_reg_model_072826_01.keras")

with open("./models/tuned_tf_avsp_reg_scaler_072826_01.pkl", "wb") as f:
    pkl.dump(scaler, f)

In [ ]:
try:
    preds_df['final_pred'] = final_preds
except:
    preds_df = test_df[[DATE_COL, TICKER_COL1, TICKER_COL2, NUM_COL, "target"]].copy()
    preds_df['final_pred'] = final_preds

In [ ]:
filename = f"./data/tf_tuned_reg_model_returns_072826_01.csv"
# pred_tickers = preds_df[[TICKER_COL1, TICKER_COL2]].drop_duplicates().reset_index(drop=True)
# ret_df = trade_strat(preds_df, pred_tickers.values, pred_col='final_pred')
# ret_df.to_csv(filename, index=False)

ret_df = pd.read_csv(filename)
ret_df[DATE_COL] = pd.to_datetime(ret_df[DATE_COL])

In [ ]:
### Get train returns
final_model = tf.keras.models.load_model("./models/tf_tuned_avsp_reg_model_072826_01.keras")

with open("./models/tuned_tf_avsp_reg_scaler_072826_01.pkl", "rb") as f:
    scaler = pkl.load(f)

X_train_scaled = scaler.transform(X_train)
final_preds = final_model.predict(X_train_scaled, verbose=0).flatten()
preds_df = train_df[[DATE_COL, TICKER_COL1, TICKER_COL2, NUM_COL, "target"]].copy()
preds_df["final_pred"] = final_preds

filename = "./data/tf_tuned_avsp_reg_model_train_returns_072826_01.csv"

pred_tickers = (preds_df[[TICKER_COL1, TICKER_COL2]].drop_duplicates().reset_index(drop=True))

ret_df = trade_strat(preds_df, pred_tickers.values, pred_col="final_pred")
ret_df.to_csv(filename, index=False)

In [ ]:
### There was some weird data corruption on the last day
plot_df = ret_df.groupby(by=DATE_COL).agg({"strat_ret" : "sum", "hold_ret" : "sum"}).iloc[:-1,:]

sns.lineplot(x=plot_df.index, y=np.log(plot_df['hold_ret']), label="Long Hold")
sns.lineplot(x=plot_df.index, y=np.log(plot_df['strat_ret']), label="Tuned Model Strategy")

plt.title("Portfolio Returns Comparison - Tuned Model Strategy v. Long Hold")
plt.ylabel("Log($ Return)")
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
### There was some weird data corruption on the last day
plot_df = ret_df.groupby(by=DATE_COL).agg({"strat_ret" : "sum", "hold_ret" : "sum", TICKER_COL1 : "count"}).rename(columns={TICKER_COL1:"num_stocks"}).iloc[:-1,:]
plot_df['base_bal'] = plot_df["num_stocks"] * START_BALANCE
plot_df["strat_pct_grwth"] = (plot_df["strat_ret"] / plot_df['base_bal'] - 1) * 100
plot_df["hold_pct_grwth"] = (plot_df["hold_ret"] / plot_df['base_bal'] - 1) * 100

sns.lineplot(x=plot_df.index, y=plot_df['hold_pct_grwth'], label="Long Hold")
sns.lineplot(x=plot_df.index, y=plot_df['strat_pct_grwth'], label="Tuned Model Strategy")

plt.title("Portfolio Returns Comparison - Tuned Model Strategy v. Long Hold")
plt.ylabel("% Return")
plt.xticks(rotation=45)
plt.legend()
plt.show()